# **任务12 图神经网络 - 图分类 | Graph Neural Network (GNN) - Graph Classification**

对整张图进行分类：对一系列有关联的数据组进行分类，类别是整个数据组的类别，关联就是有向边关系。

___

## 1. **新的层结构**

### **torch图卷积层**

区别于先前的神经网络，图卷积层（GCN）不在 `torch.nn` 子类中，而是在 `torch_geometric.nn` 需要单独下载。

它的作用是帮助不同节点信息进行交流。

In [171]:
import torch
from torch_geometric.nn import GCNConv

batch_size = 1
num_nodes = 5
input_dim = 4
output_dim = 16

# 节点特征张量
input_tensor = torch.rand((batch_size, num_nodes, input_dim))
# 边索引张量（这里只展示了10种连接，实际上每个节点都可以相互连接）
# 0→9，1→8，2→7，是一种有向连接，索引对应的是节点中的对应位置
edge_index = torch.tensor([
    [0, 1, 2, 3, 4],
    [4, 3, 2, 1, 0],
], dtype=torch.long)

conv = GCNConv(input_dim, output_dim)

output_tensor = conv(input_tensor, edge_index)

print('输入张量：', input_tensor)
print('边索引张量：', edge_index)
print('输出张量：', output_tensor.shape)

输入张量： tensor([[[0.3951, 0.4874, 0.2351, 0.0078],
         [0.6775, 0.6036, 0.0756, 0.1399],
         [0.4936, 0.4949, 0.6260, 0.3441],
         [0.0415, 0.3270, 0.7995, 0.1397],
         [0.9641, 0.6340, 0.7817, 0.0547]]])
边索引张量： tensor([[0, 1, 2, 3, 4],
        [4, 3, 2, 1, 0]])
输出张量： torch.Size([1, 5, 16])


我们看到输出结果仍然是 `batch_size=1` 个批次的 `num_nodes=5` 个节点的特征，只不过从 `input_dim=4` 变成了 `output_dim=16`，本质上只是让节点间的特征进行了交流，避免直接连接线性层导致不同特征孤立的情况。

## 2. 模型定义

对于 `torch_geometric.nn` 中的层仍然可以与 `torch.nn` 的模块联动使用

使用 `torch_geometric.utils` 的 `scatter` 方法进行图池化，将拼接在一起的多个图的节点特征，按 “所属图” 拆分并分别聚合，得到每个图的全局特征（图级特征），适配多批次并行计算。

由于图神经网络结构的特殊性，不能在输入模型之前移动到GPU设备上，如果在生成数据时转移，则可能导致显存大量占用，所以这里需要在模型中添加将张量同步转移到GPU的代码。

由于图数据的特殊性，一些激活函数需要声明使用维度，比如这次的 `softmax` 激活函数，我们使用可以声明维度的 `import torch.nn.functional as F` 类中的 `F.log_softmax(x, dim=1)` 替换。

In [172]:
import torch.nn as nn
from torch_geometric.utils import scatter  # 用于批次图池化

class Model(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Model, self).__init__()
        self.conv1 = GCNConv(input_dim, 16)
        self.conv2 = GCNConv(16, 32)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(32, num_classes)
        self.softmax = nn.Softmax()

    def forward(self, batch):
        x, edge_index, batch_idx = batch.x, batch.edge_index, batch.batch
        device = next(self.parameters()).device
        x, edge_index, batch_idx = x.to(device), edge_index.to(device), batch_idx.to(device)
        x = self.conv1(x, edge_index)
        x = self.relu(x)
        x = self.conv2(x, edge_index)
        x = self.relu(x)
        x = scatter(x, batch_idx, dim=0, reduce='mean')  # 关键：区分不同图的节点
        x = self.output_layer(x)
        return x

### 2.2 数据生成

模拟两个国家强盛程度来判断哪个国家能胜利概率，0:甲胜 1:乙胜

这里使用  `torch_geometric.data` 的 `Data` 方法，用于便捷的封装这种特殊的数据结构。使用 `torch_geometric.loader` 的 `DataLoader` 方法来划分批次。

注意 `Data(x=x, edge_index=edge_index, y=y)` 在封装的时候使用的关键字是`x, edge_index, y`，这只决定后续使用时根据对应的类成员名读取对应数据，不是硬性要求。

In [173]:
import random
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

def get_data(data_size):
    graph_datas = []
    progress = 0
    for _ in range(data_size):
        progress += 1
        a = []
        while True:
            a.append(random.randint(0, 5) + random.random())
            if sum(a) >= 500:
                break

        b = []
        while True:
            b.append(random.randint(0, 10) + random.random())
            if sum(b) >= 1000:
                break

        c = []
        while True:
            c.append(random.randint(0, 10) + random.random())
            if sum(c) >= 1000:
                break

        y = random.randint(0, 1)
        if y:
            a_0, b_0, c_0 = sum(random.choices(a, k=int(len(a) * 0.6))), sum(random.choices(b, k=int(len(b) * 0.6))), sum(random.choices(c, k=int(len(c) * 0.4))),
            a_1, b_1, c_1 = sum(random.choices(a, k=int(len(a) * 0.4))), sum(random.choices(b, k=int(len(b) * 0.4))), sum(random.choices(c, k=int(len(c) * 0.4))),
        else:
            a_1, b_1, c_1 = sum(random.choices(a, k=int(len(a) * 0.6))), sum(random.choices(b, k=int(len(b) * 0.4))), sum(random.choices(c, k=int(len(c) * 0.4))),
            a_0, b_0, c_0 = sum(random.choices(a, k=int(len(a) * 0.4))), sum(random.choices(b, k=int(len(b) * 0.4))), sum(random.choices(c, k=int(len(c) * 0.4))),

        x = []
        weights = [0.6, 0.1, 0.1, 0.05, 0.05, 0.05, 0.05]
        for weight in weights:
            a, b, c = a_0 * weight, b_0 * weight, c_0 * weight
            x.append([a, b, c])
        for weight in weights:
            a, b, c = a_1 * weight, b_1 * weight, c_1 * weight
            x.append([a, b, c])

        x = torch.tensor(x, dtype=torch.float)
        edge_index = torch.tensor([
            [0, 0, 0, 1, 1, 2, 2, 7, 7, 7, 8, 8, 9, 9],
            [1, 2, 7, 3, 4, 5, 6, 0, 8, 9, 10, 11, 12, 13]
        ], dtype=torch.long)
        y = torch.tensor([y], dtype=torch.long)

        data = Data(x=x, edge_index=edge_index, y=y)
        graph_datas.append(data)
    return graph_datas

# 训练数据
batch_size = 32
dataset = get_data(300)

cut_num = int(len(dataset)*0.7)
# 分离训练集和验证集
train_dataset = dataset[:cut_num]
val_dataset = dataset[cut_num:]
train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

for batch in train_loader:
    print("批次数据信息：", batch)
    break

批次数据信息： DataBatch(x=[448, 3], edge_index=[2, 448], y=[32], batch=[448], ptr=[33])


我们这里看到的batch维度从我们一开始设置的 `batch_size=32` 变成了 448，实际上变化是图神经网络的特点决定的，它将节点归到批次中是为了并行计算，实际上它并不会搞混不同批次的数据。

实际数据形状应该是 `x=[14, 3], edge_index=[2, 14], y=[1]` 其中 `num_nodes=14`，`input_dim=3`，一共有14条有向边，输出维度是1（二元分类的概率）。

### 2.3 模型训练

#### 2.3.1 实例化模型、损失函数、优化器

任务是分类问题，使用交叉熵损失 `nn.CrossEntropyLoss()`。

In [174]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = dataset[0].num_node_features
num_classes = 2

model = Model(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0008)

#### 2.3.2 **迭代训练**

In [175]:
epochs = 16
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        y = batch.y.to(device)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # 验证
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    model.eval()
    for batch in val_loader:
        out = model(batch)
        y = batch.y.to(device)
        loss = criterion(out, y)
        val_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += int((pred == y).sum())
        total += batch.y.size(0)
    val_loss /= len(val_loader)
    val_acc = correct / total

    if (epoch + 1) % 4 == 0:
        print(f'Epoch[{epoch + 1}] Train Loss: {train_loss}')
        print(f'\tVal Loss: {val_loss}, Val Acc: {val_acc*100:.2f}%')

Epoch[4] Train Loss: 0.4346619503838675
	Val Loss: 0.4099665880203247, Val Acc: 77.78%
Epoch[8] Train Loss: 0.24696282829557145
	Val Loss: 0.19762619833151499, Val Acc: 96.67%
Epoch[12] Train Loss: 0.18841691528047835
	Val Loss: 0.14688141892353693, Val Acc: 97.78%
Epoch[16] Train Loss: 0.1633457594684192
	Val Loss: 0.1257352208097776, Val Acc: 97.78%


## 3. 总结

这是第一个关于图结构的任务，它的计算方式不同于线性层会孤立各个特征，也解决了卷积只有相邻数据有能力进行特征交互，图神经网络能适配更具定制性的任务。